In [1]:
import findspark

In [2]:
findspark.init()

In [3]:
import py4j

In [4]:
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder.getOrCreate()

1. Count the number of employees in each city

In [22]:
df = spark.read.csv("data.csv",header=True,inferSchema=True,sep=",",encoding="utf-8")

In [23]:
df.show()

+---------+-----------+---------+-----------+---+------+-------------------------+----------+
|Education|JoiningYear|     City|PaymentTier|Age|Gender|ExperienceInCurrentDomain|LeaveOrNot|
+---------+-----------+---------+-----------+---+------+-------------------------+----------+
|Bachelors|       2017|Bangalore|          3| 34|  Male|                        0|         0|
|Bachelors|       2013|     Pune|          1| 28|Female|                        3|         1|
|Bachelors|       2014|New Delhi|          3| 38|Female|                        2|         0|
|  Masters|       2016|Bangalore|          3| 27|  Male|                        5|         1|
|  Masters|       2017|     Pune|          3| 24|  Male|                        2|         1|
|Bachelors|       2016|Bangalore|          3| 22|  Male|                        0|         0|
|Bachelors|       2015|New Delhi|          3| 38|  Male|                        0|         0|
|Bachelors|       2016|Bangalore|          3| 34|Female|    

In [24]:
from pyspark.sql.functions import count

In [26]:
employees_by_city = df.groupBy('City').agg(count('Gender').alias('count_employees'))

In [27]:
employees_by_city.show()

+---------+---------------+
|     City|count_employees|
+---------+---------------+
|Bangalore|             73|
|     Pune|             37|
|New Delhi|             40|
+---------+---------------+



2. Calculate the average ExperienceInCurrentDomain for each Education level.

In [28]:
from pyspark.sql.functions import avg

In [29]:
avg_experience_by_education_level = df.groupBy('Education').agg(avg('ExperienceInCurrentDomain').alias('avg_ExperienceInCurrentDomain'))

In [30]:
avg_experience_by_education_level.show()

+---------+-----------------------------+
|Education|avg_ExperienceInCurrentDomain|
+---------+-----------------------------+
|  Masters|            2.914285714285714|
|Bachelors|            2.452830188679245|
|      PHD|            3.111111111111111|
+---------+-----------------------------+



3. Calculate the average age of employees in each city.

In [31]:
age_by_city = df.groupBy('City').agg(avg('Age').alias('avg_age'))

In [32]:
age_by_city.show()

+---------+------------------+
|     City|           avg_age|
+---------+------------------+
|Bangalore|29.383561643835616|
|     Pune| 29.10810810810811|
|New Delhi|             29.15|
+---------+------------------+



4. How many female employees have worked 2 years or more in the company?

In [33]:
from pyspark.sql.functions import col

In [34]:
result = df.filter(
    (col("Gender") == "Female") & 
    (col("ExperienceInCurrentDomain") >= 2)
).count()

In [36]:
result

38

5. How many employees who are more than 27 years old have left the company?

In [37]:
employees_over_27 = df.filter((col('Age')>=27)&(col('LeaveOrNot') == 1)).count()

In [38]:
employees_over_27

28

6. Calculate the average age of employees who have more than 2 years of
experience


In [42]:
ave_age = df.filter(col('ExperienceInCurrentDomain')>=2).select(avg('Age'))

In [43]:
ave_age.show()

+------------------+
|          avg(Age)|
+------------------+
|29.126126126126128|
+------------------+



7. How many male employees who have more than 2 years of experience stayed in
the company?

In [44]:
df.filter((col('ExperienceInCurrentDomain')>=2)&(col('Gender') == 'Male') & (col('LeaveOrNot') == 0)).count()

54